# Visualisation — Metadata ResStock (sous-ensemble national)
Ce notebook charge un sous-ensemble du fichier national et trace quelques attributs cles.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## 1. Chargement du sous-ensemble

In [ ]:
# Colonnes qu'on veut garder (on ne charge pas les 771 colonnes)
colonnes = [
    'bldg_id',
    'in.state',
    'in.ashrae_iecc_climate_zone_2004',
    'in.geometry_floor_area',
    'in.hvac_heating_type',
    'in.hvac_cooling_type',
    'in.vintage',
    'out.electricity.total.energy_consumption..kwh',
    'out.electricity.cooling.energy_consumption..kwh',
    'out.electricity.heating.energy_consumption..kwh'
]

# read_parquet() lit le fichier parquet — on charge seulement les colonnes choisies
df_all = pd.read_parquet(r'C:\Users\user\Desktop\school\stage\upgrade0.parquet', columns=colonnes)

# sample() tire un sous-ensemble aleatoire — ici 5000 batiments sur ~550 000
# random_state=42 garantit qu'on obtient toujours le meme sous-ensemble
df = df_all.sample(n=5000, random_state=42)

# reset_index() remet un index numerique propre apres le tirage aleatoire
df = df.reset_index(drop=True)

print('Taille du sous-ensemble :', df.shape)
df.head()

## 2. Distribution par zone climatique ASHRAE

In [ ]:
# value_counts() compte le nombre de batiments par zone climatique
zones = df['in.ashrae_iecc_climate_zone_2004'].value_counts().sort_index()

# On cree une figure de taille 8x4 pouces
fig, ax = plt.subplots(figsize=(8, 4))

# bar() trace un diagramme en barres — zones.index = labels, zones.values = hauteurs
ax.bar(zones.index, zones.values, color='steelblue')

ax.set_title('Nombre de batiments par zone climatique ASHRAE')
ax.set_xlabel('Zone climatique')
ax.set_ylabel('Nombre de batiments')

# tight_layout() evite que les labels se chevauchent
plt.tight_layout()
plt.show()

## 3. Distribution des surfaces (floor area)

In [ ]:
# in.geometry_floor_area est une categorie string comme '1000-1499'
# On definit un ordre logique des intervalles pour l'axe X
ordre = [
    '0-499', '500-749', '750-999', '1000-1499', '1500-1999',
    '2000-2499', '2500-2999', '3000-3999', '4000+'
]

# value_counts() compte les batiments par tranche de surface
surfaces = df['in.geometry_floor_area'].value_counts()

# reindex() reordonne selon notre liste — les valeurs absentes deviennent 0
surfaces = surfaces.reindex([o for o in ordre if o in surfaces.index])

fig, ax = plt.subplots(figsize=(10, 4))

ax.bar(surfaces.index, surfaces.values, color='coral')

ax.set_title('Distribution des surfaces des batiments (ft²)')
ax.set_xlabel('Surface (ft²)')
ax.set_ylabel('Nombre de batiments')

# rotation=45 incline les labels de l'axe X pour qu'ils soient lisibles
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Types de chauffage HVAC

In [ ]:
# On compte les types de chauffage et on garde les 10 plus frequents
chauffage = df['in.hvac_heating_type'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 4))

# barh() trace des barres horizontales — plus lisible quand les labels sont longs
ax.barh(chauffage.index, chauffage.values, color='mediumseagreen')

ax.set_title('Types de chauffage les plus frequents (top 10)')
ax.set_xlabel('Nombre de batiments')

# invert_yaxis() met le type le plus frequent en haut
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 5. Consommation electrique totale par zone climatique

In [ ]:
# On recupere la liste des zones climatiques triees
zones_list = sorted(df['in.ashrae_iecc_climate_zone_2004'].dropna().unique())

# Pour chaque zone, on recupere les valeurs de consommation sous forme de liste
# dropna() supprime les valeurs manquantes avant de faire la boxplot
data_boxplot = [
    df[df['in.ashrae_iecc_climate_zone_2004'] == z]['out.electricity.total.energy_consumption..kwh'].dropna().values
    for z in zones_list
]

fig, ax = plt.subplots(figsize=(10, 5))

# boxplot() trace une boite a moustaches par zone
# labels associe chaque boite a son nom de zone
ax.boxplot(data_boxplot, labels=zones_list, patch_artist=True)

ax.set_title('Consommation electrique totale par zone climatique ASHRAE (kWh/an)')
ax.set_xlabel('Zone climatique')
ax.set_ylabel('Consommation (kWh/an)')

plt.tight_layout()
plt.show()